# Berlin Emergency Response
**INTRODUCTION**

---

**Zweck:** Das Projekt erklären und zeigen, was darin steckt: Fragestellung, Datenquellen, Architektur, Arbeitsweise.  
**Input:** keiner, die Zahlen unten kommen live aus `data/berlin_emergency.duckdb`.  
**Output:** Orientierung für alle weiteren Notebooks (`01`–`04`).

## Facts

| Feld | Wert |
|---|---|
| **Frage** | Welche Faktoren hängen mit der Antwortzeit des Berliner Rettungsdienstes zusammen, und wo und wann wird die Hilfsfrist verfehlt? |
| **Stakeholder** | Interessierte Öffentlichkeit, Fachpublikum aus Daten und Rettungswesen |
| **Methode** | Explorative und beschreibende Analyse auf einer dbt-Pipeline (DuckDB), Zusammenhänge statt Ursachen |
| **Hauptdatenquelle** | Open Data der Berliner Feuerwehr (`BF-Open-Data`, CC BY 4.0) |
| **Zielgrößen** | Antwortzeit in Sekunden, Anteil Einsätze innerhalb der Hilfsfrist (offiziell und eigene Näherung) |
| **Nicht Teil des Projekts** | Bewertung von Einsatzkräften, Kausalaussagen, Prognosemodelle |

## Research Question

> Welche Faktoren hängen mit der Antwortzeit des Berliner Rettungsdienstes zusammen (Zeit, Ort, Einsatzart und Dringlichkeit, Sondersituationen, gegebenenfalls Wetter), und wo und wann wird die Hilfsfrist verfehlt?

Wir nutzen die öffentlich zugänglichen Daten der Berliner Feuerwehr und reichern sie schrittweise mit weiteren Daten an. Gezeigt werden **Zusammenhänge, keine Ursachen**: Antwortzeiten hängen an Standortdichte, Verkehr, Bebauung und Einsatzaufkommen, nicht an der Leistung einzelner Teams.

### Mission Types and Time Goals

Die Feuerwehr arbeitet in drei Einsatzarten mit **unterschiedlichen Vorgaben**. Jede Gesamtbetrachtung in diesem Projekt wird deshalb nach Einsatzart aufgeschlüsselt.

| Einsatzart | Vorgabe (Schutzziel) | Gemessen als |
|---|---|---|
| **Rettungsdienst** | erstes Rettungsmittel innerhalb von 10 Minuten, Erreichungsgrad 90 % | Zeit bis zum ersten Fahrzeug |
| **Brandbekämpfung** | 14 Funktionen innerhalb von 15 Minuten; Erreichungsgrad 90 % (Klasse A), 50 % (Klasse B) | Zeit bis zum ersten Löschfahrzeug, zur ersten Drehleiter, bis 14 Einsatzkräfte vor Ort sind |
| **Technische Hilfeleistung** | keine Frist und kein Erreichungsgrad vereinbart; die BF weist nur die durchschnittlich erreichte Zeit aus | Antwortzeit-Statistik |

Die Hilfsfrist läuft laut BF vom Beginn der Notrufabfrage bis zum Eintreffen der ersten Einsatzkräfte. Quelle: [Berliner Feuerwehr in Zahlen](https://www.berliner-feuerwehr.de/ueber-uns/berliner-feuerwehr-in-zahlen-2024-1/), ergänzt um [Senatsantwort 2019](https://pardok.parlament-berlin.de/starweb/adis/citat/VT/19/SchrAnfr/S19-19457.pdf) und die BF-Jahresberichte 2022 und 2023. Den Gesetzestext haben wir nicht geprüft.

### Dimensions

| Dimension | Frage | Notebook | Datenbasis |
|---|---|---|---|
| Zeit | Wie verändern sich Antwortzeiten über Jahre, Saison und Wochentag? | `03_analysis_time` | Einsätze, Tagesreihe |
| Ort | Wo sind Antwortzeiten länger, wo wird die Hilfsfrist verfehlt? | `03_analysis_geo` | Bezirke (Einsätze), Regionen (Jahresaggregat) |
| Einsatzart und Mittel | Hängt die Zeit an Einsatzart (Rettungsdienst, Brand, technische Hilfe), Dringlichkeit, Einsatzmitteln? Jeweils getrennt ausgewertet | `03_analysis_mission_type` | Einsätze |
| Sonderereignisse | Pandemie, Silvester, Strukturbrüche in der Definition | `03_analysis_special_events` | Einsätze, Tagesreihe, `seed_events` |
| Wetter, Feiertage | Gibt es Effekte von Witterung, Feiertagen und Ferien? | `03_analysis_weather` | Anreicherung (geplant) |

### Hypotheses

Vermutungen, die wir in den Notebooks prüfen, nicht Annahmen:

* Vermutung: Antwortzeiten sind in dünn bebauten Außenbezirken länger als im Zentrum.
* Vermutung: Die Hilfsfrist wird in einsatzstarken Zeiten (Sommer, Wochenenden) häufiger verfehlt.
* Vermutung: Der Abstand zur nächsten Wache hängt stärker mit Quote und Antwortzeit zusammen als der Abstand zum Zentrum (erste Hinweise in `01_exploration_stations`, für den Rettungsdienst).
* Belegt (nicht mehr Vermutung): Der Anstieg der offiziellen Quote von 2024 auf 2025 hängt an der geänderten Definition von "kritisch" (Notfallkategorien seit dem 25.03.2025), nicht an schnelleren Einsätzen; zusätzlich verschiebt der RTW-B (ab Januar 2023) die Hilfsfrist.
* Vermutung: Sondersituationen (Pandemiejahre, Jahreswechsel) zeigen sich in Einsatzzahlen stärker als in Antwortzeiten.

### Limits of the Data

* Keine Uhrzeit und keine Einsatzkoordinaten, nur Tag und Bezirk (12). "Wo" heißt: Bezirk im Zeitverlauf, Region pro Jahr.
* Regionaldaten (Bezirksregion, Planungsraum, Prognoseraum) nur als Jahresaggregat und nur ab 2024.
* Bei einem Teil der Einsätze fehlt die Antwortzeit; Einsätze haben keine ID, Duplikate sind nicht erkennbar.
* Die Grundgesamtheit "kritischer" Einsätze ändert sich am 25.03.2025 (siehe `seed_events`); Jahresvergleiche der offiziellen Quote sind eingeschränkt. Vergleichbar über die ganze Zeit ist die Gruppe RD1+RD2.

## Data Sources

Alle Einsatzdaten stammen aus dem Repository [`Berliner-Feuerwehr/BF-Open-Data`](https://github.com/Berliner-Feuerwehr/BF-Open-Data) (CC BY 4.0). Upstream hat keinen Tagesendpunkt: täglich wird die Datei des laufenden Jahres neu geladen, Vorjahre einmalig (`scripts/ingest.py`).

### Overview

| Quelle | Granularität | Inhalt | Warum wir sie nutzen | Was für uns wichtig ist |
|---|---|---|---|---|
| **Mission_Data** (`mission_data_set_open_data_<Jahr>.csv`) | ein Einsatz, 2018 bis heute | Tag, Einsatzart, Dispatch-Code und Stufe (A–E, O), Bezirk, Antwortzeit, Einsatzmittel (`units_*`), Notarzt, First Responder | einzige Quelle auf Einsatzebene, trägt Zeitverlauf und Abhängigkeiten | `response_time`, `mission_type`, `dispatchcode_criticality`, `mission_location_district`, `units_first_type` |
| **Daily_Data** (`BFw_mission_data_daily.csv`) | Stadt je Tag, ab 2018 | Einsatzzahlen nach Kategorie, Statistik der Antwortzeiten, Rettungsdienst-Notfallkategorien RD1 bis RD5 (laut BF sind RD1 und RD2 "critical") | Gegenprobe zu Mission_Data, Tagesreihe für Sonderereignisse | `mission_count_*`, `response_time_ems_critical_*` |
| **Regional_Data** (`<Jahr>/BFw_{planning_room,district_area,prediction_area}_data_<Jahr>.csv`) | LOR-Region je Jahr, ab 2024 | Einsatzzahlen, Antwortzeit-Statistik, **Hilfsfrist: berechnet und erreicht** | einzige Quelle mit offizieller Hilfsfrist-Quote und geografischer Tiefe | `*_timegoal_computed`, `*_timegoal_reached`, drei LOR-Ebenen mit stabilen IDs |
| **Turnout_Times** (`crew_turnout_times_<Jahr>_q<N>.csv`, `_current`) | Wache je Quartal | Alarmierungen und mittlere Ausrückdauer für RTW, NEF, LHF, DLK, ELW | Anteil der Ausrückzeit an der Gesamtzeit, Blick auf Wachen | `mittlere_ausrueckedauer_*`, `wache_name` |
| **Einsatzcodes** (`Dispatchcodes/*.xlsx`, Stand 21.05.2026) | Code | vollständige AMPDS-Codes des Rettungsdienstes (Kategorie, Stufe, laufende Nummer) mit Notfallkategorie RD1 bis RD5 und Einsatzmittelaufgebot | Namen der Kategorien und Zuordnung Kategorie und Stufe zu Notfallkategorien (`seed_dispatch_codes`, `seed_dispatch_code_map`) | deckt nur Kategorien bis 34 ab; Einzeleinsätze tragen nur Kategorie und Stufe, die Notfallkategorie ist daraus nicht eindeutig |
| **Open-Data-Seite der Berliner Feuerwehr** (Diagramme) | Diagramm | Einsatzzahlen nach Art, Eintreffzeiten (Notfallrettung, Reanimation, Brand), Notfallkategorien, Prognoseräume | Referenz für unsere Ergebnisse und Quelle der Annotationen (RTW-B, Notfallkategorien, Stürme) | Stichtag 25.03.2025 der Notfallkategorien |
| **Feuerwehr-Standorte und Einsatzbereiche** (Geoportal Berlin, WFS `feuerwehr`, Stand 2024) | Standort | Wachen (Berufsfeuerwehr, Freiwillige, Rettungswachen) mit Koordinaten, Bezirk und Einsatzbereich, sechs Einsatzbereiche als Flächen | Koordinaten der Wachen: Abstand zur nächsten Wache, Zuordnung der Ausrückzeiten zu Bezirken | `wach_nr` entspricht `wache_nummer` in den Ausrückzeiten; dl-de-zero-2.0 |
| **LOR-Polygone** (Geoportal Berlin, WFS `su_lor`, Stand 01.01.2021) | 3 Ebenen | Flächen der Planungsräume, Bezirksregionen, Prognoseräume | Karte; IDs stimmen mit den BF-Regionen überein | CC BY 3.0 DE, Quellenvermerk Pflicht |
| **Phase 2 (geplant)** | Tag, Region | DWD Open Data (Wetter), Feiertage und Ferien, Kiez Data und regionale Kennzahlen, Berlin Open Data (Bevölkerung, Stadtstruktur, Geodaten) | Anreicherung (Stufe 1: Wetter, Kalender; Stufe 2: Struktur) | Datensätze noch zu klären |

Nicht genutzt: `Daily_Data/BFw_call_data_*` (Notrufzahlen) und `KV_Data` (Übergaben an die Kassenärztliche Vereinigung Berlin). Die Disclaimer-Datei weist darauf hin, dass Notrufzahlen bei Rückfallbetrieb unvollständig sind.

### Response Time and Time Goal

* **Antwortzeit** (`response_time`): Zeit von der Einsatzanlage bis zum Eintreffen des ersten Fahrzeugs, in Sekunden. BF dokumentiert im Upstream-README, dass für Notruf und Disposition zusätzlich zwei Minuten angesetzt werden.
* **Hilfsfrist laut BF:** Zeit vom Beginn der Notrufabfrage bis zum Eintreffen der ersten Einsatzkräfte.
* **Hilfsfrist, offiziell:** Anteil der kritischen Rettungsdiensteinsätze, bei denen die Frist erreicht wurde (`timegoal_reached` geteilt durch `timegoal_computed`), nur in den Regionaldaten.
* **Hilfsfrist, eigene Näherung (nur Rettungsdienst):** Anteil der Rettungsdiensteinsätze der Dispatch-Stufen C, D und E mit Antwortzeit unter einer Schwelle (dbt-Var `timegoal_seconds`, je Einsatzgruppe, derzeit nur Rettungsdienst, Vermutung). C/D/E kommt der offiziellen Gruppe RD1+RD2 am nächsten (`01_exploration_daily`). Brandbekämpfung (14 Funktionen) und technische Hilfe (keine Frist) haben keine Schwelle auf einzelne Einsätze.

## Architecture

### Medallion Layers

```
BF-Open-Data (GitHub)   LOR-Polygone (WFS)
        │                      │
        ▼                      │
scripts/ingest.py              │   idempotent, harter Fehler statt Fallback
        │
        ▼
BRONZE   raw_*                 unveränderte Rohdaten (VARCHAR), _partition, _loaded_at
        │  dbt
        ▼
SILVER   stg_* → int_*         typisiert, benannt, Bezirke und Flags, LOR-Hierarchie
        │
        ▼
GOLD     fct_* / dim_*         analysefertige Marts (table, incremental)
        │  scripts/export_parquet.py
        ▼
SERVING  public/app/data/*.parquet    nur das, was die App liest (+ insights.json aus 04_insights)
```

### dbt Components

| Komponente | Wo | Wie wir sie anwenden | Warum |
|---|---|---|---|
| **Sources + Freshness** | `dbt/models/staging/sources.yml` | sechs Raw-Tabellen; Freshness über `_loaded_at` für Einsätze und Tagesreihe (warn nach 2, error nach 4 Tagen) | trennt Ingestion von Transformation, meldet veraltete Daten |
| **Seeds** | `dbt/seeds/` | `seed_districts` (Bezirkscode und Name), `seed_stations` (Wachen mit Koordinaten), `seed_dispatch_codes` und `seed_dispatch_code_map` (BF-Codetabelle), `seed_events` (belegte Ereignisse mit Quelle) | kleine, versionierte Referenzdaten statt Hardcodes im SQL |
| **Staging** (view) | `dbt/models/staging/` | 1:1 zur Quelle: umbenennen, typisieren, keine Geschäftslogik; Macros `to_int`, `stg_regional` | Typen und Namen an genau einer Stelle |
| **Macros, Jinja, Vars** | `dbt/macros/`, `dbt/dbt_project.yml` | `is_within_timegoal` (Schwelle je Einsatzgruppe), `safe_pct`, Var `timegoal_seconds`; Schleifen über Spaltenlisten | Schwelle zentral änderbar, kein Copy-Paste |
| **Intermediate** (view) | `dbt/models/intermediate/` | Bezirk-Join, Einsatzgruppe, Stufe, Hilfsfrist-Flag; die drei Regional-Ebenen vereint, Hierarchie aus den IDs | Geschäftslogik getrennt von Typisierung und Marts |
| **Marts** (table, incremental) | `dbt/models/marts/` | `fct_timegoal_region_yearly`, `fct_missions_daily_district` (incremental: `delete+insert` auf `mission_date`, Filter über `loaded_at`), `fct_missions_daily_citywide`, `fct_turnout_times_quarterly`, `dim_district`, `dim_region` | große Faktentabelle nicht täglich komplett neu bauen |
| **Tests** | `schema.yml`, `dbt/tests/singular/` | generisch (`unique`, `not_null`, `relationships`, `accepted_values`), `dbt_utils` (Kombination, Wertebereich, Ausdruck), singulär (Abgleich Regional gegen Einsätze, keine Zukunftsdaten, Ebenen stimmig) | Datenfehler fallen vor der App auf |
| **Snapshots, Docs, Exposures, Contracts, Groups, Versions** | S5 | folgt | SCD2-Historie der Regionaldaten, Lineage, Governance der Marts |

### Automation

Heute laufen die Schritte über `make ingest`, `make dbt-build`, `make export`. In S4 übernimmt GitHub Actions täglich: Ingestion, Freshness, Build, Export, Commit der App-Daten.

### Repository Structure

Engineering, Analysis, Science und die App leben im selben Repository, jede Arbeit in einem eigenen Ordner:

| Ordner | Bereich | Inhalt |
|---|---|---|
| `dbt/` | Data Engineering | das vollständige dbt-Projekt (Modelle, Macros, Seeds, Tests, Profil); läuft selbstständig |
| `scripts/` | Data Engineering | Ingestion und einmalige Abrufe (`ingest.py`, `fetch_*`, `export_parquet.py`) |
| `notebooks/` | Analysis und Science | nummeriert in Erzählreihenfolge (`00_` bis `0x_`); Modelltraining würde als `05_`/`06_` folgen |
| `src/berlin_emergency_response/` | Analysis und Science | Python-Paket: Konfiguration, Pfade, räumliche Helfer, später Features und Modelle |
| `src/app/` | Ergebnis | Code der Streamlit-App (Seiten, Hilfsfunktionen) |
| `public/` | Veröffentlichung | Web-Root: Hub und Views, `public/app/data/` mit allen Daten der Live-App |
| `data/` | alle | Rohdaten, DuckDB, später Modelle (nicht versioniert) |
| `tests/`, `docs/` | alle | pytest für Python; Plan, Konzept, Datenwörterbuch |

## Workflow

### Notebook Map

| Notebook | Aufgabe | Output |
|---|---|---|
| `00_introduction` | Fragestellung, Quellen, Architektur | diese Orientierung |
| `01_exploration_<quelle>` (missions, daily, regional, turnout, geo, stations) | je Quelle durchleuchten (`wgnd.inspect`) und ein Fazit: Inhalt, Nutzen, Grenzen | Befunde je Quelle |
| `02_preparation` | die Pipeline im Detail: Bereinigung, Zusammenführung, Aufbereitung, Automatisierung mit dbt | nachvollziehbarer Weg Bronze zu Gold |
| `03_analysis_<dimension>` | gezielte Untersuchungen auf den Gold-Tabellen | belegte Muster |
| `04_insights` | die Diagramme der BF-Open-Data-Seite nachgebaut, ergänzt um unsere Befunde je Einsatzart; Erkenntnisse zusammenführen, Schlussfolgerungen, Empfehlungen | Grundlage der App (`public/app/data/insights.json`) |

### Conventions

| Variable | Zustand |
|---|---|
| `con` | schreibgeschützte DuckDB-Verbindung (`read_only=True`) |
| `df_raw` | Auszug aus einer `raw_*`-Tabelle, unverändert |
| `df_edit` | bereinigt oder aggregiert, immer mit `.copy()` aus `df_raw` |
| `df_final` | Ergebnis für Charts und Insights |

Charts nach `public/img/`, Auswertungen für die App nach `public/app/data/`. Pfade über `PATHS`.

## Setup

In [1]:
from berlin_emergency_response.notebook import *
import duckdb

setup_plotting()
con = duckdb.connect(str(PATHS["db"]), read_only=True)

✓  wgnd theme activated (matplotlib · seaborn)

### Rows per Layer

In [2]:
df_tables = con.sql("select table_name from information_schema.tables where table_schema = 'main'").df()

def layer_of(name: str) -> str:
    for prefix, layer in [('seed_', '0 seed'), ('raw_', '1 bronze'), ('stg_', '2 silver'), ('int_', '2 silver'), ('fct_', '3 gold'), ('dim_', '3 gold')]:
        if name.startswith(prefix):
            return layer
    return ''

df_tables['layer'] = df_tables['table_name'].map(layer_of)
df_layers = df_tables[df_tables['layer'] != ''].sort_values(['layer', 'table_name']).copy()
df_layers['rows'] = [con.sql(f'select count(*) from {t}').fetchone()[0] for t in df_layers['table_name']]
show_df(df_layers[['layer', 'table_name', 'rows']].reset_index(drop=True))

,layer,table_name,rows
0,0 seed,seed_dispatch_code_map,123
1,0 seed,seed_dispatch_codes,35
2,0 seed,seed_districts,12
3,0 seed,seed_events,6
4,0 seed,seed_stations,102
5,1 bronze,raw_daily_mission_data,3195
6,1 bronze,raw_mission_data,4461202
7,1 bronze,raw_regional_district_area,429
8,1 bronze,raw_regional_planning_room,1626
9,1 bronze,raw_regional_prediction_area,174
